In [ ]:

# HiDMamba Notebook 00: PTB-XL Data Preprocessing
# Phase 0 | Output artifact: ptbxl-processed-v1

import ast
import json
import os
import warnings
from collections import defaultdict
from pathlib import Path

import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import wfdb
except ImportError:
    print("[bootstrap] Installing wfdb...")
    _pip('wfdb>=4.0.0')
    import wfdb

try:
    from tqdm.auto import tqdm
except ImportError:
    _pip('tqdm')
    from tqdm.auto import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
np.random.seed(42)



# CONSTANTS

SAMPLING_FREQ     = 100
TARGET_LENGTH     = 1000
N_LEADS           = 12
CONFIDENCE_THRESH = 50.0

TRAIN_FOLDS = list(range(1, 9))
VAL_FOLD    = 9
TEST_FOLD   = 10

SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("  HiDMamba -Notebook 00: PTB-XL Data Preprocessing")
print("=" * 72)
print(f"  Output directory    : {OUTPUT_DIR}")
print(f"  Sampling frequency  : {SAMPLING_FREQ} Hz")
print(f"  Target length       : {TARGET_LENGTH} samples  ({TARGET_LENGTH // SAMPLING_FREQ} s)")
print(f"  Confidence threshold: >= {CONFIDENCE_THRESH}%")
print(f"  Train / Val / Test  : folds {TRAIN_FOLDS} / {VAL_FOLD} / {TEST_FOLD}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1/9 -Locate PTB-XL root
# ─────────────────────────────────────────────────────────────────────────────
def find_ptbxl_root(search_base='/kaggle/input'):
    base = Path(search_base)
    if not base.exists():
        raise FileNotFoundError(f"Search base does not exist: {search_base}")
    for db_csv in base.rglob('ptbxl_database.csv'):
        parent = db_csv.parent
        if (parent / 'scp_statements.csv').exists() and (parent / 'records100').is_dir():
            return parent
    for db_csv in base.rglob('ptbxl_database.csv'):
        return db_csv.parent
    raise FileNotFoundError(
        "\n[ERROR] PTB-XL not found under /kaggle/input.\n"
        "Add it via '+ Add Input' in the Kaggle sidebar."
    )

PTBXL_ROOT  = find_ptbxl_root()
DB_CSV      = PTBXL_ROOT / 'ptbxl_database.csv'
SCP_CSV     = PTBXL_ROOT / 'scp_statements.csv'
RECORDS_DIR = PTBXL_ROOT / 'records100'

print(f"\n[OK] PTB-XL root: {PTBXL_ROOT}")
assert DB_CSV.exists(),      f"Missing: {DB_CSV}"
assert SCP_CSV.exists(),     f"Missing: {SCP_CSV}"
assert RECORDS_DIR.is_dir(), f"Missing directory: {RECORDS_DIR}"

print("\n" + "-" * 60)
print("SECTION 1/9  - Loading metadata files")
print("-" * 60)

db  = pd.read_csv(DB_CSV,  index_col='ecg_id')
scp = pd.read_csv(SCP_CSV, index_col=0)

print(f"  ptbxl_database.csv  : {len(db):,} records")
print(f"  scp_statements.csv  : {len(scp):,} SCP codes")
print(f"  Database columns    : {list(db.columns)}")
print(f"  SCP columns         : {list(scp.columns)}")

for col in ('strat_fold', 'filename_lr', 'scp_codes'):
    if col not in db.columns:
        raise ValueError(f"Required column '{col}' missing from ptbxl_database.csv.")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2/9 -Build label hierarchy
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 2/9  - Building label hierarchy")
print("-" * 60)

def _find_col(df, options):
    for opt in options:
        if opt in df.columns:
            return opt
    return None

diag_col = _find_col(scp, ['diagnostic', 'Diagnostic'])
dc_col   = _find_col(scp, ['diagnostic_class',    'Diagnostic_class'])
ds_col   = _find_col(scp, ['diagnostic_subclass', 'Diagnostic_subclass'])

if diag_col is None:
    raise ValueError(
        f"Cannot find 'diagnostic' column in scp_statements.csv.\n"
        f"Available: {list(scp.columns)}"
    )

print(f"  diagnostic col       : '{diag_col}'")
print(f"  diagnostic_class col : '{dc_col}'")
print(f"  diagnostic_sub col   : '{ds_col}'")

diag_mask = scp[diag_col].fillna(0).astype(float).astype(int) == 1
diag_scp  = scp[diag_mask].copy()
print(f"  Diagnostic SCP codes : {len(diag_scp):,}")

# Map each diagnostic code → (superclass, subclass)
code_to_superclass = {}
code_to_subclass   = {}

for code, row in diag_scp.iterrows():
    sc_raw = row.get(dc_col) if dc_col else None
    sc = str(sc_raw).strip() if pd.notna(sc_raw) else None
    if not sc or sc not in SUPERCLASSES:
        continue
    code_to_superclass[code] = sc

    if ds_col:
        sb_raw = row.get(ds_col)
        sb = str(sb_raw).strip() if pd.notna(sb_raw) else None
        if sb and sb.lower() not in ('nan', '', 'none'):
            code_to_subclass[code] = sb
        elif sc == 'NORM':
            code_to_subclass[code] = 'NORM'
    else:
        code_to_subclass[code] = str(code)

# Build hierarchy dict
hierarchy = defaultdict(set)
for code, sc in code_to_superclass.items():
    sb = code_to_subclass.get(code)
    if sb:
        hierarchy[sc].add(sb)

if 'NORM' not in hierarchy or not hierarchy['NORM']:
    hierarchy['NORM'] = {'NORM'}

hierarchy = {sc: sorted(list(sbs)) for sc, sbs in hierarchy.items()}

# ── Full subclass list (including any that share a name with a superclass) ───
ALL_SUBCLASSES_FULL = []
for sc in SUPERCLASSES:
    for sb in hierarchy.get(sc, []):
        if sb not in ALL_SUBCLASSES_FULL:
            ALL_SUBCLASSES_FULL.append(sb)

# ── KEY FIX: remove subclass names that are already superclass column names.
#    NORM and STTC appear in BOTH lists.  Having them twice creates duplicate
#    DataFrame columns, causing  df['NORM'].sum()  to return a Series
#    instead of a scalar, which breaks  int(...)  with a TypeError.
#    The superclass column already captures these labels perfectly, so
#    dropping the duplicate subclass entry loses zero information.
# ────────────────────────────────────────────────────────────────────────────
UNIQUE_SUBCLASSES = [sb for sb in ALL_SUBCLASSES_FULL if sb not in SUPERCLASSES]

# Final column list -guaranteed unique
ALL_LABEL_COLS = SUPERCLASSES + UNIQUE_SUBCLASSES   # e.g. 5 + 21 = 26

print(f"\n  Hierarchy (5 superclasses):")
for sc in SUPERCLASSES:
    subs = hierarchy.get(sc, [])
    print(f"    {sc:6s} ({len(subs):2d}): {subs}")

print(f"\n  Full subclass list  ({len(ALL_SUBCLASSES_FULL):2d}): {ALL_SUBCLASSES_FULL}")
print(f"  Unique subclasses   ({len(UNIQUE_SUBCLASSES):2d}): {UNIQUE_SUBCLASSES}")
print(f"  (Dropped as duplicates of superclasses: "
      f"{[s for s in ALL_SUBCLASSES_FULL if s in SUPERCLASSES]})")
print(f"\n  Total label columns : {len(ALL_LABEL_COLS)}  "
      f"(5 super + {len(UNIQUE_SUBCLASSES)} unique sub)")

with open(OUTPUT_DIR / 'label_hierarchy.json', 'w') as f:
    json.dump(hierarchy, f, indent=2, ensure_ascii=False)
print(f"  Saved -> {OUTPUT_DIR / 'label_hierarchy.json'}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3/9 -Parse SCP codes → multi-hot label matrix
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 3/9  - Parsing scp_codes -> multi-hot label matrix")
print("-" * 60)
print(f"  Confidence filter: only codes with likelihood >= {CONFIDENCE_THRESH}%")


def _parse_scp_codes(raw):
    if pd.isna(raw) or str(raw).strip() in ('', 'nan', 'NaN'):
        return {}
    try:
        parsed = ast.literal_eval(str(raw))
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}


records_list = []

for ecg_id, row in tqdm(db.iterrows(), total=len(db),
                         desc='  Parsing labels', ncols=80, unit='rec'):
    scp_dict = _parse_scp_codes(row['scp_codes'])

    qualifying = {
        code: conf
        for code, conf in scp_dict.items()
        if float(conf) >= CONFIDENCE_THRESH and code in code_to_superclass
    }

    # ── KEY FIX: sub_vec only contains UNIQUE_SUBCLASSES (no NORM / STTC).
    #    Previously sub_vec had NORM and STTC as keys, and entry.update(sub_vec)
    #    would silently overwrite the values correctly set by super_vec for those
    #    two fields.  Now they are cleanly separated: superclass labels live
    #    exclusively in super_vec; non-overlapping subclass labels in sub_vec.
    # ─────────────────────────────────────────────────────────────────────────
    super_vec = {sc: 0 for sc in SUPERCLASSES}
    sub_vec   = {sb: 0 for sb in UNIQUE_SUBCLASSES}

    for code in qualifying:
        sc = code_to_superclass.get(code)
        sb = code_to_subclass.get(code)
        if sc:
            super_vec[sc] = 1
        # Only set sub_vec if the subclass name is NOT a superclass name
        # (NORM->NORM and STTC->STTC are already handled by super_vec above)
        if sb and sb in sub_vec:
            sub_vec[sb] = 1

    entry = {
        'ecg_id'     : ecg_id,
        'patient_id' : row.get('patient_id', np.nan),
        'strat_fold' : int(row['strat_fold']),
        'filename_lr': str(row.get('filename_lr', '')).strip(),
        'n_diag'     : len(qualifying),
    }
    entry.update(super_vec)
    entry.update(sub_vec)
    records_list.append(entry)

meta_df = pd.DataFrame(records_list).set_index('ecg_id')

n_with_label = (meta_df[SUPERCLASSES].sum(axis=1) > 0).sum()
n_no_label   = (meta_df[SUPERCLASSES].sum(axis=1) == 0).sum()
n_multi      = (meta_df[SUPERCLASSES].sum(axis=1) > 1).sum()

print(f"\n  Records parsed              : {len(meta_df):,}")
print(f"  With >=1 diagnostic label   : {n_with_label:,}")
print(f"  Multi-superclass records    : {n_multi:,}")
print(f"  All-zero label rows (kept)  : {n_no_label:,}")

print(f"\n  Superclass distribution:")
for sc in SUPERCLASSES:
    n   = int(meta_df[sc].values.sum())
    pct = 100.0 * n / len(meta_df)
    print(f"    {sc:6s}: {n:6,}  ({pct:5.1f}%)")

print(f"\n  Unique subclass distribution:")
for sb in UNIQUE_SUBCLASSES:
    n   = int(meta_df[sb].values.sum())
    pct = 100.0 * n / len(meta_df)
    print(f"    {sb:22s}: {n:6,}  ({pct:5.1f}%)")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4/9 -Fold split verification
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 4/9  - Fold split verification")
print("-" * 60)

fold_counts = meta_df['strat_fold'].value_counts().sort_index()
print(f"  Per-fold record counts:\n{fold_counts.to_string()}")

patient_overlap_count = 0
if 'patient_id' in meta_df.columns:
    train_pts = set(
        meta_df.loc[meta_df['strat_fold'].isin(TRAIN_FOLDS), 'patient_id'].dropna().unique()
    )
    test_pts = set(
        meta_df.loc[meta_df['strat_fold'] == TEST_FOLD, 'patient_id'].dropna().unique()
    )
    overlap = train_pts & test_pts
    patient_overlap_count = len(overlap)
    if overlap:
        print(f"\n  [!] WARNING: {len(overlap)} patients in BOTH train and test!")
    else:
        print(f"\n  [OK] Phase 0 criterion: zero patient overlap between train and test.")

fold_split = {str(k): int(v) for k, v in meta_df['strat_fold'].to_dict().items()}
with open(OUTPUT_DIR / 'fold_split.json', 'w') as f:
    json.dump(fold_split, f)
print(f"  Saved -> {OUTPUT_DIR / 'fold_split.json'}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5/9 -Load ECG signals at 100 Hz (using wfdb)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 5/9  - Loading ECG signals at 100 Hz")
print("-" * 60)
print(f"  Records directory   : {RECORDS_DIR}")
print(f"  Target shape        : ({N_LEADS}, {TARGET_LENGTH})")

ecg_ids_loaded = []
signals_list   = []
failed_records = []

for ecg_id, row in tqdm(meta_df.iterrows(), total=len(meta_df),
                          desc='  Loading signals', ncols=80, unit='rec'):
    filename_lr = row['filename_lr'].strip()
    for _ext in ('.hea', '.dat'):
        if filename_lr.endswith(_ext):
            filename_lr = filename_lr[:-len(_ext)]

    record_path = str(PTBXL_ROOT / filename_lr)

    try:
        record = wfdb.rdrecord(record_path)
        signal = record.p_signal          # (T, 12)  mV

        if signal is None:
            raise ValueError("wfdb returned None for p_signal.")
        if signal.ndim != 2:
            raise ValueError(f"Expected 2-D array, got shape {signal.shape}.")
        if signal.shape[1] != N_LEADS:
            raise ValueError(f"Expected {N_LEADS} leads, got {signal.shape[1]}.")

        if signal.shape[0] > TARGET_LENGTH:
            signal = signal[:TARGET_LENGTH, :]
        elif signal.shape[0] < TARGET_LENGTH:
            pad    = np.zeros((TARGET_LENGTH - signal.shape[0], N_LEADS), dtype=np.float32)
            signal = np.vstack([signal, pad])

        signal = signal.T.astype(np.float32)     # -> (12, 1000)

        if not np.isfinite(signal).all():
            signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)

        ecg_ids_loaded.append(ecg_id)
        signals_list.append(signal)

    except Exception as exc:
        failed_records.append((ecg_id, str(exc)))

print(f"\n  Successfully loaded : {len(signals_list):,}")
print(f"  Failed to load      : {len(failed_records):,}")

if failed_records:
    print(f"  First 5 failures:")
    for fid, ferr in failed_records[:5]:
        print(f"    ecg_id={fid}: {ferr}")
    pd.DataFrame(failed_records, columns=['ecg_id', 'error']).to_csv(
        OUTPUT_DIR / 'load_failures.csv', index=False
    )

if len(signals_list) == 0:
    raise RuntimeError("No signals loaded. Check that records100/ contains WFDB files.")

ecg_id_array  = np.array(ecg_ids_loaded, dtype=np.int64)
signals_array = np.stack(signals_list, axis=0)     # (N, 12, 1000)

print(f"\n  signals_array shape  : {signals_array.shape}")
print(f"  Memory (float32)     : {signals_array.nbytes / 1e6:.1f} MB")

labels_df  = meta_df.loc[ecg_id_array].copy()
train_mask = labels_df['strat_fold'].isin(TRAIN_FOLDS).values
val_mask   = (labels_df['strat_fold'] == VAL_FOLD).values
test_mask  = (labels_df['strat_fold'] == TEST_FOLD).values

print(f"\n  Loaded train records  : {train_mask.sum():,}")
print(f"  Loaded val records    : {val_mask.sum():,}")
print(f"  Loaded test records   : {test_mask.sum():,}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6/9 -Per-lead z-score normalisation (train fold only)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 6/9  - Per-lead z-score normalisation")
print("-" * 60)
print("  Statistics computed on TRAIN folds 1-8 only, then applied to ALL.")

train_sigs = signals_array[train_mask]      # (N_train, 12, 1000)

lead_mean = train_sigs.mean(axis=(0, 2))    # (12,)
lead_std  = train_sigs.std( axis=(0, 2))    # (12,)

degenerate = lead_std < 1e-8
if degenerate.any():
    print(f"  [!] Degenerate leads: {[LEAD_NAMES[i] for i in np.where(degenerate)[0]]} -> std set to 1.0")
lead_std = np.where(degenerate, 1.0, lead_std)

print(f"\n  Per-lead raw stats (train fold):")
print(f"  {'Lead':6s}  {'Mean (mV)':>12}  {'Std (mV)':>10}")
for i, name in enumerate(LEAD_NAMES):
    print(f"  {name:6s}  {lead_mean[i]:12.5f}  {lead_std[i]:10.5f}")

signals_norm = (
    (signals_array - lead_mean[np.newaxis, :, np.newaxis]) /
     lead_std      [np.newaxis, :, np.newaxis]
).astype(np.float32)

tm = float(signals_norm[train_mask].mean())
ts = float(signals_norm[train_mask].std())
print(f"\n  Post-normalisation check (train fold):")
print(f"    Global mean  ~0 : {tm:.6f}  {'OK' if abs(tm) < 0.01 else 'FAIL'}")
print(f"    Global std   ~1 : {ts:.4f}   {'OK' if abs(ts - 1.0) < 0.05 else 'FAIL'}")
print(f"    No NaN          : {not np.isnan(signals_norm).any()}")
print(f"    No Inf          : {not np.isinf(signals_norm).any()}")

norm_stats = {
    'lead_mean'       : lead_mean.tolist(),
    'lead_std'        : lead_std.tolist(),
    'lead_names'      : LEAD_NAMES,
    'n_leads'         : N_LEADS,
    'target_length'   : TARGET_LENGTH,
    'sampling_freq_hz': SAMPLING_FREQ,
    'fit_on_folds'    : TRAIN_FOLDS,
    'apply_formula'   : 'signal_norm = (signal_mV - lead_mean) / lead_std   shape (12,)',
}
with open(OUTPUT_DIR / 'norm_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)
print(f"  Saved -> {OUTPUT_DIR / 'norm_stats.json'}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7/9 -Class weights  (BCEWithLogitsLoss pos_weight)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 7/9  - Computing class weights (train fold only)")
print("-" * 60)

# Use .values.sum() to always get a numpy scalar regardless of pandas version.
# This is robust even if column selection somehow returns a 2-D structure.
train_labels_df = labels_df[train_mask][ALL_LABEL_COLS]
n_train         = len(train_labels_df)

print(f"  Train label matrix shape: {train_labels_df.shape}")
print(f"  Unique column count     : {len(train_labels_df.columns.unique())}")

# Verify no duplicate columns survived (safety assertion)
dupes = [c for c in ALL_LABEL_COLS if ALL_LABEL_COLS.count(c) > 1]
if dupes:
    raise RuntimeError(
        f"Duplicate column names detected in ALL_LABEL_COLS: {dupes}\n"
        "This should have been fixed by the deduplication step above."
    )

class_weights = {}
for col in ALL_LABEL_COLS:
    col_data = train_labels_df[col]
    # col_data must be a Series; if it's a DataFrame we have a dup-column bug
    if isinstance(col_data, pd.DataFrame):
        raise RuntimeError(
            f"Column '{col}' returned a DataFrame instead of a Series. "
            "Duplicate column names exist -check ALL_LABEL_COLS construction."
        )
    n_pos = int(col_data.values.sum())    # .values gives numpy array, .sum() is scalar
    n_neg = n_train - n_pos
    class_weights[col] = float(n_neg / max(n_pos, 1))

with open(OUTPUT_DIR / 'class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)

top5 = sorted(class_weights.items(), key=lambda x: -x[1])[:5]
print(f"\n  Weight range : {min(class_weights.values()):.1f}x – {max(class_weights.values()):.1f}x")
print(f"  Top 5 most imbalanced:")
for lbl, w in top5:
    print(f"    {lbl:25s}: pos_weight = {w:.1f}x")
print(f"  Saved -> {OUTPUT_DIR / 'class_weights.json'}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8/9 -Save all output artifacts
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 8/9  - Saving output artifacts")
print("-" * 60)

np.save(OUTPUT_DIR / 'ptbxl_100hz.npy',   signals_norm)
np.save(OUTPUT_DIR / 'ecg_id_order.npy',  ecg_id_array)
sz = os.path.getsize(OUTPUT_DIR / 'ptbxl_100hz.npy') / 1e6
print(f"  [saved] ptbxl_100hz.npy     shape={signals_norm.shape}  {sz:.1f} MB")
print(f"  [saved] ecg_id_order.npy    shape={ecg_id_array.shape}")

labels_out = labels_df[['strat_fold'] + ALL_LABEL_COLS].copy()
labels_out.index.name = 'ecg_id'
labels_out.to_csv(OUTPUT_DIR / 'labels_multihot.csv')
print(f"  [saved] labels_multihot.csv shape={labels_out.shape}  "
      f"(fold + {len(SUPERCLASSES)} super + {len(UNIQUE_SUBCLASSES)} sub)")
print(f"  [exists] label_hierarchy.json, norm_stats.json, class_weights.json, fold_split.json")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9/9 -Sanity checks  (Phase 0 criteria)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("SECTION 9/9  - Sanity checks (Phase 0 criteria)")
print("-" * 60)

chk_signals = np.load(OUTPUT_DIR / 'ptbxl_100hz.npy')
chk_labels  = pd.read_csv(OUTPUT_DIR / 'labels_multihot.csv', index_col='ecg_id')
chk_ids     = np.load(OUTPUT_DIR / 'ecg_id_order.npy')

with open(OUTPUT_DIR / 'label_hierarchy.json') as f:
    chk_hier = json.load(f)
with open(OUTPUT_DIR / 'norm_stats.json') as f:
    chk_norm = json.load(f)

# Verify no duplicate columns in the saved CSV
dup_cols_in_csv = [c for c in chk_labels.columns if list(chk_labels.columns).count(c) > 1]

checks = {
    "Signal shape is (N, 12, 1000)"         : chk_signals.shape == (len(ecg_id_array), N_LEADS, TARGET_LENGTH),
    "Label rows == signal rows"              : len(chk_labels) == len(chk_signals),
    "ecg_id_order length == signal rows"    : len(chk_ids) == len(chk_signals),
    "No NaN in signals"                     : not bool(np.isnan(chk_signals).any()),
    "No Inf in signals"                     : not bool(np.isinf(chk_signals).any()),
    "Train mean close to 0  (|m| < 0.01)"  : abs(float(chk_signals[train_mask].mean())) < 0.01,
    "Train std  close to 1  (|s-1| < 0.05)": abs(float(chk_signals[train_mask].std()) - 1.0) < 0.05,
    "All strat_fold values in 1-10"         : bool(chk_labels['strat_fold'].isin(range(1, 11)).all()),
    "No duplicate columns in CSV"           : len(dup_cols_in_csv) == 0,
    "All label values are binary {0, 1}"    : set(chk_labels[ALL_LABEL_COLS].values.ravel().tolist()).issubset({0, 1}),
    "Hierarchy has 5 superclasses"          : len(chk_hier) == 5,
    "norm_stats has lead_mean and lead_std" : 'lead_mean' in chk_norm and 'lead_std' in chk_norm,
    "Zero patient overlap train/test"       : patient_overlap_count == 0,
}

all_pass = True
for name, result in checks.items():
    marker = "OK" if result else "FAIL"
    print(f"  [{marker}] {name}")
    if not result:
        all_pass = False

print(f"\n  -> {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED -review above'}")


# ─────────────────────────────────────────────────────────────────────────────
# Visualisations
# ─────────────────────────────────────────────────────────────────────────────
print("\nGenerating plots...")

fig = plt.figure(figsize=(20, 18))
fig.suptitle(
    f'HiDMamba -Notebook 00: PTB-XL Preprocessing\n'
    f'{len(chk_signals):,} records · {SAMPLING_FREQ} Hz · {N_LEADS} leads',
    fontsize=13, fontweight='bold', y=1.002,
)
time_ax = np.arange(TARGET_LENGTH) / SAMPLING_FREQ

# Plot 1: sample ECG
ax1 = fig.add_subplot(4, 2, (1, 2))
ex_idx, ex_sc = None, 'NORM'
for sc_try in SUPERCLASSES:
    idxs = np.where(train_mask & (labels_df[sc_try].values == 1))[0]
    if len(idxs) > 0:
        ex_idx = int(idxs[0])
        ex_sc  = sc_try
        break

if ex_idx is not None:
    ecg_ex = chk_signals[ex_idx]
    offset = 5.0
    for li in range(N_LEADS):
        ax1.plot(time_ax, ecg_ex[li] + li * offset, color='#1a3a5c', lw=0.65, alpha=0.9)
    ax1.set_yticks([li * offset for li in range(N_LEADS)])
    ax1.set_yticklabels(LEAD_NAMES, fontsize=8)
    ax1.set_xlabel('Time (s)', fontsize=9)
    ax1.set_xlim(0, TARGET_LENGTH / SAMPLING_FREQ)
    ax1.set_title(
        f'Sample ECG  (ecg_id={ecg_id_array[ex_idx]}, class={ex_sc})'
        ' -z-score normalised, leads offset for readability', fontsize=10,
    )
    ax1.grid(True, alpha=0.18, ls='--')

# Plot 2: superclass distribution
ax2 = fig.add_subplot(4, 2, 3)
sc_counts = [int(chk_labels[sc].values.sum()) for sc in SUPERCLASSES]
colors    = ['#4CAF50', '#F44336', '#2196F3', '#FF9800', '#9C27B0']
bars = ax2.bar(SUPERCLASSES, sc_counts, color=colors, edgecolor='white', lw=1.1)
for bar, cnt in zip(bars, sc_counts):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + max(sc_counts) * 0.015,
             f'{cnt:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.set_title('Superclass Distribution', fontsize=10)
ax2.set_ylabel('Records', fontsize=9)
ax2.grid(axis='y', alpha=0.3, ls='--')

# Plot 3: fold distribution
ax3 = fig.add_subplot(4, 2, 4)
fold_vals   = sorted(fold_counts.index)
fold_colors = ['#4CAF50' if f in TRAIN_FOLDS else
               '#FF9800' if f == VAL_FOLD    else '#F44336' for f in fold_vals]
ax3.bar(fold_vals, fold_counts[fold_vals].values, color=fold_colors, edgecolor='white')
ax3.set_xticks(fold_vals)
ax3.set_xlabel('Fold index', fontsize=9)
ax3.set_ylabel('Records', fontsize=9)
ax3.set_title('Fold Distribution  (green=train 1-8  |  orange=val 9  |  red=test 10)', fontsize=10)
ax3.grid(axis='y', alpha=0.3, ls='--')

# Plot 4: subclass distribution
ax4 = fig.add_subplot(4, 2, (5, 6))
# Show all subclasses including NORM and STTC (via their superclass columns)
all_sb_for_plot = ALL_SUBCLASSES_FULL   # original 23, including NORM and STTC
sb_counts = [int(chk_labels[sb].values.sum()) for sb in all_sb_for_plot]
yp = np.arange(len(all_sb_for_plot))
ax4.barh(yp, sb_counts, color='#008080', edgecolor='white', lw=0.7, alpha=0.85)
ax4.set_yticks(yp)
ax4.set_yticklabels(all_sb_for_plot, fontsize=7.5)
ax4.set_xlabel('Records', fontsize=9)
ax4.set_title('Subclass Distribution  (all 23 original subclasses)', fontsize=10)
ax4.grid(axis='x', alpha=0.3, ls='--')

# Plot 5: per-lead raw mean & std
ax5 = fig.add_subplot(4, 2, 7)
x = np.arange(N_LEADS)
w = 0.35
ax5.bar(x - w/2, lead_mean, w, label='Mean (mV)', color='steelblue', alpha=0.85, edgecolor='white')
ax5.bar(x + w/2, lead_std,  w, label='Std  (mV)', color='tomato',    alpha=0.85, edgecolor='white')
ax5.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax5.set_xticks(x)
ax5.set_xticklabels(LEAD_NAMES, fontsize=8)
ax5.set_xlabel('Lead', fontsize=9)
ax5.set_ylabel('mV', fontsize=9)
ax5.set_title('Per-Lead Raw Mean & Std (train fold) -> z-score normalisation values', fontsize=10)
ax5.legend(fontsize=9)
ax5.grid(axis='y', alpha=0.3, ls='--')

# Plot 6: class imbalance
ax6 = fig.add_subplot(4, 2, 8)
cw_labels = list(class_weights.keys())
cw_vals   = [class_weights[k] for k in cw_labels]
bar_clrs  = ['#4CAF50' if lbl in SUPERCLASSES else '#008080' for lbl in cw_labels]
ax6.bar(range(len(cw_labels)), cw_vals, color=bar_clrs, edgecolor='white', lw=0.6, alpha=0.85)
ax6.set_xticks(range(len(cw_labels)))
ax6.set_xticklabels(cw_labels, rotation=60, ha='right', fontsize=7)
ax6.set_ylabel('pos_weight  (neg/pos)', fontsize=9)
ax6.set_title('Class Imbalance per Label  (BCEWithLogitsLoss pos_weight)', fontsize=10)
ax6.grid(axis='y', alpha=0.3, ls='--')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preprocessing_summary.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print(f"  Saved -> {OUTPUT_DIR / 'preprocessing_summary.png'}")


# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing report
# ─────────────────────────────────────────────────────────────────────────────
report_lines = [
    "=" * 72,
    "  HiDMamba -Notebook 00: PTB-XL Preprocessing Report",
    "=" * 72,
    "",
    "-- Configuration -------------------------------------------------------",
    f"  PTB-XL root              : {PTBXL_ROOT}",
    f"  Sampling frequency       : {SAMPLING_FREQ} Hz  (records100/)",
    f"  Target signal length     : {TARGET_LENGTH} samples  ({TARGET_LENGTH//SAMPLING_FREQ} s)",
    f"  SCP confidence threshold : >= {CONFIDENCE_THRESH}%",
    f"  Label columns            : {len(ALL_LABEL_COLS)} unique  "
    f"(5 super + {len(UNIQUE_SUBCLASSES)} sub; NORM and STTC counted once each)",
    f"  Fold split               : train {TRAIN_FOLDS} | val {VAL_FOLD} | test {TEST_FOLD}",
    f"  Normalisation            : per-lead z-score, fit on train folds only",
    "",
    "-- Record Counts -------------------------------------------------------",
    f"  Records in database      : {len(db):,}",
    f"  Successfully loaded      : {len(signals_list):,}",
    f"  Failed to load           : {len(failed_records):,}",
    f"  With >=1 diag label      : {n_with_label:,}",
    f"  Multi-superclass         : {n_multi:,}",
    f"  All-zero (kept)          : {n_no_label:,}",
    "",
    "-- Fold Distribution (loaded records) ----------------------------------",
    f"  Train (folds 1-8)        : {train_mask.sum():,}",
    f"  Val   (fold  9)          : {val_mask.sum():,}",
    f"  Test  (fold 10)          : {test_mask.sum():,}",
    f"  Patient overlap (train/test): {patient_overlap_count}  "
    f"{'[PASS]' if patient_overlap_count == 0 else '[FAIL]'}",
    "",
    "-- Superclass Distribution (all loaded records) -------------------------",
]
for sc in SUPERCLASSES:
    n   = int(chk_labels[sc].values.sum())
    pct = 100.0 * n / len(chk_labels)
    report_lines.append(f"  {sc:8s}: {n:7,} / {len(chk_labels):,}  ({pct:5.1f}%)")

report_lines += [
    "",
    "-- Subclass Distribution (all 23 original subclasses) ------------------",
    f"  {'Subclass':22s}  {'Count':>7}  {'%':>6}  {'pos_weight':>10}",
]
for sb in ALL_SUBCLASSES_FULL:
    n   = int(chk_labels[sb].values.sum())
    pct = 100.0 * n / len(chk_labels)
    w   = class_weights.get(sb, class_weights.get(sb, 0.0))
    note = "  [= superclass col]" if sb in SUPERCLASSES else ""
    report_lines.append(f"  {sb:22s}  {n:7,}  {pct:6.1f}%  {w:10.1f}x{note}")

report_lines += [
    "",
    "-- Per-Lead Normalisation Stats (train fold) ----------------------------",
    f"  {'Lead':6s}  {'Raw Mean (mV)':>14}  {'Raw Std (mV)':>13}",
]
for i, name in enumerate(LEAD_NAMES):
    report_lines.append(f"  {name:6s}  {lead_mean[i]:14.5f}  {lead_std[i]:13.5f}")

report_lines += [
    "",
    "-- Phase 0 Sanity Checks -----------------------------------------------",
]
for name, result in checks.items():
    report_lines.append(f"  [{'OK' if result else 'FAIL'}] {name}")

report_lines += [
    "",
    "-- Output Artifacts ----------------------------------------------------",
    f"  ptbxl_100hz.npy         : shape {chk_signals.shape}  float32  normalised",
    f"  ecg_id_order.npy        : shape {chk_ids.shape}  int64   row->ecg_id",
    f"  labels_multihot.csv     : {chk_labels.shape}  "
    f"(fold + {len(ALL_LABEL_COLS)} unique label cols)",
    f"  label_hierarchy.json    : 5 superclasses -> subclasses",
    f"  norm_stats.json         : per-lead mean/std",
    f"  class_weights.json      : pos_weight per label",
    f"  fold_split.json         : ecg_id -> fold",
    "",
    "-- Next Step -----------------------------------------------------------",
    "  Version /kaggle/working as Kaggle Dataset 'ptbxl-processed-v1'.",
    "=" * 72,
]

report_text = "\n".join(report_lines)
print("\n" + report_text)

with open(OUTPUT_DIR / 'preprocessing_report.txt', 'w') as f:
    f.write(report_text)


# ─────────────────────────────────────────────────────────────────────────────
# Final file listing
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  NOTEBOOK 00 COMPLETE")
print("=" * 72)
print(f"\n  {'File':<44}  {'Size':>9}")
print(f"  {'-'*44}  {'-'*9}")
for fp in sorted(OUTPUT_DIR.glob('*')):
    size_str = f"{os.path.getsize(fp)/1e6:.2f} MB"
    print(f"  {fp.name:<44}  {size_str:>9}")

[bootstrap] Installing wfdb...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.0 MB/s eta 0:00:00
  HiDMamba — Notebook 00: PTB-XL Data Preprocessing
  Output directory    : /kaggle/working
  Sampling frequency  : 100 Hz
  Target length       : 1000 samples  (10 s)
  Confidence threshold: >= 50.0%
  Train / Val / Test  : folds [1, 2, 3, 4, 5, 6, 7, 8] / 9 / 10

[OK] PTB-XL root: /kaggle/input/datasets/iaadidev/ptb-xl-a-large-public-ecg-dataset-1-0-3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3

------------------------------------------------------------
SECTION 1/9  —  Loading metadata files
------------------------------------------------------------
  ptbxl_database.csv  : 21,799 records
  scp_statements.csv  : 71 SCP codes
  Database columns    : ['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_op

  Parsing labels:   0%|                              | 0/21799 [00:00<?, ?rec/s]


  Records parsed              : 21,799
  With >=1 diagnostic label   : 20,373
  Multi-superclass records    : 4,390
  All-zero label rows (kept)  : 1,426

  Superclass distribution:
    NORM  :  9,438  ( 43.3%)
    MI    :  4,134  ( 19.0%)
    STTC  :  5,078  ( 23.3%)
    CD    :  4,891  ( 22.4%)
    HYP   :  2,258  ( 10.4%)

  Unique subclass distribution:
    AMI                   :  2,559  ( 11.7%)
    IMI                   :  2,242  ( 10.3%)
    LMI                   :    122  (  0.6%)
    PMI                   :     13  (  0.1%)
    ISCA                  :    934  (  4.3%)
    ISCI                  :    391  (  1.8%)
    ISC_                  :  1,260  (  5.8%)
    NST_                  :    557  (  2.6%)
    CLBBB                 :    536  (  2.5%)
    CRBBB                 :    540  (  2.5%)
    ILBBB                 :     76  (  0.3%)
    IRBBB                 :  1,115  (  5.1%)
    IVCD                  :    787  (  3.6%)
    LAFB/LPFB             :  1,796  (  8.2%)
    WPW  

  Loading signals:   0%|                             | 0/21799 [00:00<?, ?rec/s]


  Successfully loaded : 21,799
  Failed to load      : 0

  signals_array shape  : (21799, 12, 1000)
  Memory (float32)     : 1046.4 MB

  Loaded train records  : 17,418
  Loaded val records    : 2,183
  Loaded test records   : 2,198

------------------------------------------------------------
SECTION 6/9  —  Per-lead z-score normalisation
------------------------------------------------------------
  Statistics computed on TRAIN folds 1-8 only, then applied to ALL.

  Per-lead raw stats (train fold):
  Lead       Mean (mV)    Std (mV)
  I           -0.00173     0.15690
  II          -0.00149     0.16023
  III          0.00023     0.16348
  aVR          0.00159     0.13586
  aVL         -0.00093     0.13886
  aVF         -0.00060     0.14150
  V1           0.00016     0.23043
  V2          -0.00095     0.33133
  V3          -0.00155     0.32733
  V4          -0.00136     0.29181
  V5          -0.00080     0.26717
  V6          -0.00240     0.27583

  Post-normalisation check (train f